# Exploratory Data Analysis + Data Cleaning #

### 1. Import Libraries ###

In [ ]:
# reading/cleaning data
import pandas as pd
# visualizations
from matplotlib import pyplot as plt
import seaborn as sns

### 2. Load/Clean Data

In [ ]:
# load advanced stats
def deduplicate_traded_players(df):
    '''
    Ensure that for traded players, we only keep their 2TM/3TM rows
    '''
    # Mark rows that are combined totals (2TM, 3TM, etc.)
    df["is_combined"] = df["Team"].str.match(r"^\dTM$", na=False)
    df = df.sort_values("is_combined")
    df = df.drop_duplicates(subset=["Player-additional", "season"], keep="last")
    df = df.drop(columns="is_combined")
    return df

dfs = []
for year in range(2021, 2025):
    df = pd.read_csv(f"data/advanced_{year}.csv")
    df["season"] = year
    dfs.append(df)

player_data = pd.concat(dfs, ignore_index=True)
player_data = deduplicate_traded_players(player_data)
player_data.drop(columns=["Awards"])

In [ ]:
# read the PPG values for all players from 2021-2025, and merge them with player_data
ppg_dfs = []

for year in range(2021, 2026):  # one extra year, so we capture the PPG for the next year 
    pg = pd.read_csv(f"data/pergame_{year}.csv")[["Player-additional", "Team", "PTS"]]
    pg["season"] = year
    pg["is_combined"] = pg["Team"].str.match(r"^\dTM$", na=False)
    pg = pg.sort_values("is_combined")
    pg = pg.drop_duplicates(subset=["Player-additional", "season"], keep="last")
    pg = pg.drop(columns=["is_combined", "Team"])
    ppg_dfs.append(pg)

ppg = pd.concat(ppg_dfs, ignore_index=True)

# Merge PPG into advanced df
player_data = player_data.merge(ppg, on=["Player-additional", "season"], how="left")
player_data.drop(columns=['Awards', 'Player-additional', 'Rk'], inplace=True)

In [ ]:
print(f'Dataset Shape: {player_data.shape}')
player_data.head(10)

### 3. Dataset Overview

In [ ]:
player_data.info()

In [ ]:
player_data.describe().round(2)

In [ ]:
# check for duplicate rows
print(f"Duplicate rows: {player_data.duplicated().sum()}")

In [ ]:
player_data.columns

### 5. Target Variable (PTS) Distribution

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(13,4))
color = 'mediumseagreen'

# histogram of Next Season PPG
axes[0].hist(player_data.PTS, bins = 40, color = color, edgecolor='white', alpha = 0.85)
axes[0].axvline(player_data.PTS.mean(), color = 'red', linestyle = '--', label = f"Mean: {player_data['PTS'].mean():.2f}")
axes[0].set_title('Next Season PPG Distribution')
axes[0].legend()

# box plot
sns.boxplot(y = player_data.PTS, ax = axes[1], color = 'lightblue')
axes[1].set_title('PTS Box Plot')
plt.tight_layout(); plt.show()

print(f'Skewness: {player_data.PTS.skew():.2f}')
print(f'Range: {player_data.PTS.min():.2f} → {player_data.PTS.max():.2f}')

### 6. Univariate — Numeric Features

In [ ]:
numeric_cols = [c for c in player_data.columns if player_data[c].dtype == 'float64' 
                and c != 'PTS'] # all columns in player_data, without PTS (the target var)
ncols = 3; nrows = -(-len(numeric_cols) // ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 16))
axes = axes.flatten()

# histograms for each numeric column
for i, col in enumerate(numeric_cols):
    axes[i].hist(player_data[col], color = color, edgecolor='white', bins = 20)
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('count')

for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout(); plt.show()

In [ ]:
# scatter plot comparing each numeric col to PTS
fig, axes = plt.subplots(nrows, ncols, figsize=(15,16))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    axes[i].scatter(player_data[col], player_data.PTS, color = color)
    axes[i].set_title(f'{col} vs. PTS')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('PTS')

for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout(); plt.show()

### 7. Correlation Analysis

In [ ]:
# correlation heatmap for numeric features
corr_df = player_data.copy()
corr = corr_df[numeric_cols + ['PTS']].corr()
plt.figure(figsize=(15,9))
sns.heatmap(corr, annot = True, fmt = ".2f", cmap = 'coolwarm')
plt.title("Numeric Feature Correlation Heatmap")
plt.tight_layout(); plt.show()

In [ ]:
# correlation of all numeric features with PTS
corr_with_pts = corr["PTS"].drop("PTS").sort_values()

plt.figure(figsize=(9, 7))
plt.barh(corr_with_pts.index, corr_with_pts.values, color = ["coral" if v > 0 else "steelblue" for v in corr_with_pts.values])
plt.axvline(x=0, color="black", linewidth=0.8, linestyle="--")
plt.axvline(x=0.6, color='black', linewidth=0.8, linestyle="--")
plt.title("Feature Correlation with PTS")
plt.xlabel("Correlation Coefficient"); plt.ylabel("Features")
plt.tight_layout(); plt.show()

print(f"Based on the correlation heatmap and barplot, the features with the highest correlation (i.e. coefs ≥ 0.6) are {corr_with_pts[corr_with_pts.abs() >= 0.6].sort_values(ascending = False)}")

### 8. EDA Findings

1. PTS is very right skewed, which is to be expected(very few players will be scoring lots). This indicates that a linear regression may not be the most accurate model for predictions, and a tree-based model may perform better
2. MP and GS are the strongest predictors of PTS, but they are also very correlated with each other, meaning that only one of them should be included as a final feature
3. Age is not a strong predictor, as it is not strongly correlated with PTS
4. While the amount of playing time (MP, GS) is strongly correlated with PTS, TS% is not, which indicates that the *opportunity* to shoot is more important than the scoring efficiency of the player
5. USG%, OBPM, PER, and AST% have strong correlation with PTS, which suggests that players with larger offensive roles tend to score more
6. There is significant multicollinearity in the model, such as the with the aforementioned MP and GS, WS and OWS/DWS, and BPM/OBPM and PER. This supports the aforementioned suggestion that a linear model may struggle significantly, while a tree-based model may perform significantly better

Based on this, a possible list of best features is likely  MP, USG%, OBPM, AST%

### 9. Saving final CSV for model building

In [ ]:
player_data.to_csv("data/player_data_cleaned.csv", index = False)
print("Saved data/player_data_cleaned.csv")